# 02 — First cost model results: how long can a driver queue?

**Question:** at what queue time does CNG conversion stop paying off for a ride-hailing driver and a keke rider?

This notebook runs the daily cost model in `src/cost_model.py` for two vehicles, three scenarios and a range of driver
earnings, then finds the **break-even queue time**: the queue length at which the fuel saving is used up by lost
earnings and loan repayments.

**Every input value is read from `data/raw/model_inputs.csv`** (sources in `docs/sources.md`). The notebook only
decides *which column* (low / mid / high) each scenario uses. Values whose confidence is `Assumption` are labelled
as such wherever they appear.

## 1. Setup

Same pattern as notebook 01: find the project root, add `src/` to the import path, then import our own modules.
`cost_model` is imported as `cm` so calls read as `cm.fuel_saving_per_day(...)`, which makes it obvious where each
function lives.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
import load_data          # noqa: E402
import cost_model as cm   # noqa: E402

FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Show money without scientific notation in tables
pd.options.display.float_format = "{:,.2f}".format

## 2. Load the inputs

`get_input` looks up one row of `model_inputs.csv` by parameter and vehicle and returns its low / mid / high values
as numbers, plus the source IDs. It raises an error if the row is missing or duplicated, so a typo in a parameter
name fails loudly instead of quietly producing a wrong answer. Blank cells become `None`.

In [2]:
inputs = load_data.read_csv_safe(load_data.RAW_DIR / "model_inputs.csv")
LEVELS = ["low", "mid", "high"]


def get_input(parameter, vehicle):
    """Return {'low': x, 'mid': y, 'high': z, 'source': 'S..'} for one row of model_inputs.csv."""
    match = inputs[(inputs["parameter"] == parameter) & (inputs["vehicle"] == vehicle)]
    if len(match) != 1:
        raise ValueError(f"Expected 1 row for {parameter} / {vehicle}, found {len(match)}")
    row = match.iloc[0]
    values = {level: float(row[level]) if row[level].strip() else None for level in LEVELS}
    values["source"] = row["source_ids"] or row["confidence"]   # e.g. 'S18', or 'Assumption'
    return values


# Inputs that vary by scenario
petrol_price    = get_input("petrol_price_national", "all")   # NGN per litre
cng_price       = get_input("cng_price", "car/bus")           # NGN per scm (only light-vehicle price; keke uses it too)
interest_pct    = get_input("loan_interest_rate", "all")      # percent per year
tenor           = get_input("loan_tenor", "all")              # years
queue_time      = get_input("queue_time_per_refuel", "all")   # hours per refuel
conv_car        = get_input("conversion_cost", "car")         # NGN
conv_keke       = get_input("conversion_cost", "keke")        # NGN; mid = midpoint of 100k-650k (see CSV note)

# Vehicle spends (one reported value each, in the mid column)
spend_car       = get_input("ridehail_daily_fuel_petrol", "car")    # NGN per day, S21
spend_keke      = get_input("keke_daily_fuel_cost_petrol", "keke")  # NGN per day, S19

# Assumption rows: the same value in every column, so we always read 'mid'
working_days    = get_input("working_days_per_year", "all")["mid"]           # days per year (Assumption)
spend_ref_price = get_input("petrol_price_at_spend_reports", "all")["mid"]   # NGN/litre (Assumption, S19;S21)
refuels_per_day = get_input("refuels_per_day", "all")["mid"]                 # per day (Assumption)
efficiency      = get_input("efficiency_ratio", "all")["mid"]                # ratio (Low, S20;S05)

pd.Series({
    "Working days per year (Assumption)": working_days,
    "Petrol price when spends were reported, NGN/L (Assumption)": spend_ref_price,
    "Refuels per day (Assumption)": refuels_per_day,
    "Efficiency ratio, km/scm ÷ km/L (Low confidence, C4)": efficiency,
}, name="value")

Working days per year (Assumption)                             312.00
Petrol price when spends were reported, NGN/L (Assumption)   1,000.00
Refuels per day (Assumption)                                     1.00
Efficiency ratio, km/scm ÷ km/L (Low confidence, C4)             1.00
Name: value, dtype: float64

## 3. Vehicle profiles and the petrol-spend rescaling

Both daily petrol spends were reported when petrol cost **about ₦1,000 per litre** (`petrol_price_at_spend_reports`,
an Assumption row citing S19 and S21). Petrol costs more in every scenario now, so we rescale:

> rescaled spend = reported spend × scenario petrol price ÷ petrol price when the spend was reported

**This assumes each driver buys the same number of litres per day as when the spend was reported**, i.e. covers the
same distance. If drivers have cut back on driving as prices rose, this overstates today's spend and the fuel saving.
Distance per day is a survey item (`daily_distance` is a Gap row in `model_inputs.csv`).

The keke conversion cost now has a mid value in the CSV: ₦375,000, the midpoint of the ₦100k–₦650k range.

In [3]:
vehicles = {
    "Ride-hailing car": {
        "spend_reported": spend_car["mid"],
        "spend_source": spend_car["source"],
        "conversion_cost": conv_car,
    },
    "Keke": {
        "spend_reported": spend_keke["mid"],
        "spend_source": spend_keke["source"],
        "conversion_cost": conv_keke,
    },
}

pd.DataFrame({
    name: {
        "Reported petrol spend (NGN/day)": v["spend_reported"],
        "Spend source": v["spend_source"],
        "Conversion cost low (NGN)": v["conversion_cost"]["low"],
        "Conversion cost mid (NGN)": v["conversion_cost"]["mid"],
        "Conversion cost high (NGN)": v["conversion_cost"]["high"],
        "Conversion cost source": v["conversion_cost"]["source"],
    }
    for name, v in vehicles.items()
})

,Ride-hailing car,Keke
Reported petrol spend (NGN/day),"20,000.00","15,000.00"
Spend source,S21,S19
Conversion cost low (NGN),"300,000.00","100,000.00"
Conversion cost mid (NGN),"1,300,000.00","375,000.00"
Conversion cost high (NGN),"1,700,000.00","650,000.00"
Conversion cost source,S42;S40;S41,S43;S04


## 4. Scenarios

Three scenarios, each defined by **which direction helps or hurts CNG** rather than by column name:

| Input | CNG-favourable | Central | CNG-unfavourable |
|---|---|---|---|
| Petrol price | highest | mid | lowest |
| CNG price | lowest | mid | highest |
| Conversion cost | lowest | mid | highest |
| Loan interest rate | lowest | mid | highest |
| Loan tenor | **longest** | mid | **shortest** |
| Queue time | shortest | mid | longest |

Loan tenor is the one that runs "backwards": a *longer* loan spreads the same cost over more days, so the daily
repayment is *smaller*, which helps CNG.

`SCENARIO_LEVELS` below records only *which column* each scenario reads. The values themselves all come from the CSV,
so updating `model_inputs.csv` updates every scenario without touching this notebook.

The favourable and unfavourable scenarios are **bookends**, not forecasts: every input sits at its extreme at the same
time, which is unlikely in practice. The real answer is likely to sit between them.

Assumption rows (refuels per day, efficiency ratio, working days, spend reference price) are the same in every scenario.

In [4]:
SCENARIO_LEVELS = {
    "CNG-favourable": {
        "petrol_price": "high", "cng_price": "low", "conversion_cost": "low",
        "interest_rate": "low", "tenor_years": "high", "queue_hours": "low",
    },
    "Central": {
        "petrol_price": "mid", "cng_price": "mid", "conversion_cost": "mid",
        "interest_rate": "mid", "tenor_years": "mid", "queue_hours": "mid",
    },
    "CNG-unfavourable": {
        "petrol_price": "low", "cng_price": "high", "conversion_cost": "high",
        "interest_rate": "high", "tenor_years": "low", "queue_hours": "high",
    },
}
SCENARIOS = list(SCENARIO_LEVELS)   # keeps this order in tables: favourable, central, unfavourable

# Map each scenario input to the CSV row it comes from
INPUT_ROWS = {
    "petrol_price": petrol_price,
    "cng_price": cng_price,
    "interest_rate": interest_pct,
    "tenor_years": tenor,
    "queue_hours": queue_time,
}

scenarios = {}
for name, levels in SCENARIO_LEVELS.items():
    s = {key: INPUT_ROWS[key][levels[key]] for key in INPUT_ROWS}
    s["interest_rate"] = s["interest_rate"] / 100          # cost_model wants a decimal: 17.5% -> 0.175
    s["conversion_level"] = levels["conversion_cost"]      # conversion cost differs by vehicle, so store the column
    scenarios[name] = s

# Display table: every value actually used, interest shown as a percentage for readability
display = pd.DataFrame({
    name: {
        "Petrol price (NGN/L)": s["petrol_price"],
        "CNG price (NGN/scm)": s["cng_price"],
        "Loan interest (% per yr)": s["interest_rate"] * 100,
        "Loan tenor (years)": s["tenor_years"],
        "Queue time (hrs per refuel)": s["queue_hours"],
        "Conversion cost, car (NGN)": conv_car[s["conversion_level"]],
        "Conversion cost, keke (NGN)": conv_keke[s["conversion_level"]],
    }
    for name, s in scenarios.items()
})
display

,CNG-favourable,Central,CNG-unfavourable
Petrol price (NGN/L),"1,596.00","1,400.00","1,051.00"
CNG price (NGN/scm),230.00,318.00,380.00
Loan interest (% per yr),15.00,17.50,20.00
Loan tenor (years),3.00,2.00,1.00
Queue time (hrs per refuel),1.50,4.00,6.00
"Conversion cost, car (NGN)","300,000.00","1,300,000.00","1,700,000.00"
"Conversion cost, keke (NGN)","100,000.00","375,000.00","650,000.00"


## 5. Earnings per hour: a sensitivity range

Net earnings per hour (after owner remittance and app commission) are **not yet known**; this is a survey item. Rather
than guess one number, we test **₦1,000 to ₦5,000 per hour in steps of ₦500**.

> ⚠️ This is a **sensitivity range pending survey data**, not an estimate of what drivers earn.

Earnings matter because they set the price of an hour in the queue: the more a driver earns, the more each hour of
waiting costs them, and the shorter the queue they can afford.

In [5]:
EARNINGS_PER_HOUR = list(range(1_000, 5_001, 500))   # NGN/hour; range() stops before 5,001, so 5,000 is included
EARNINGS_PER_HOUR

[1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000]

## 6. Run the model

One loop over every combination: 2 vehicles × 3 scenarios × 9 earnings levels = 54 rows. Each row calls the
`cost_model` functions in the order the daily budget works: fuel saving, then queue cost, loan cost, net benefit, and
finally the break-even queue time.

We collect plain dictionaries in a list and build the DataFrame once at the end, which is simpler and faster than
adding rows to a DataFrame one at a time.

In [6]:
rows = []
for vehicle_name, v in vehicles.items():
    for scenario_name, s in scenarios.items():
        # Rescale the reported spend to this scenario's petrol price (same litres per day)
        petrol_spend = v["spend_reported"] * s["petrol_price"] / spend_ref_price
        conversion_cost = v["conversion_cost"][s["conversion_level"]]

        # These two don't depend on earnings, so work them out once per scenario
        fuel_saving = cm.fuel_saving_per_day(petrol_spend, s["petrol_price"], s["cng_price"], efficiency)
        loan_cost = cm.loan_cost_per_day(conversion_cost, s["interest_rate"], s["tenor_years"], working_days)

        for earnings in EARNINGS_PER_HOUR:
            queue_cost = cm.queue_cost_per_day(refuels_per_day, s["queue_hours"], earnings)
            rows.append({
                "vehicle": vehicle_name,
                "scenario": scenario_name,
                "earnings_per_hour": earnings,
                "petrol_spend_per_day": petrol_spend,
                "fuel_saving": fuel_saving,
                "queue_cost": queue_cost,
                "loan_cost": loan_cost,
                "net_benefit": cm.net_benefit_per_day(fuel_saving, queue_cost, loan_cost),
                "break_even_queue_hours": cm.break_even_queue_hours(
                    fuel_saving, loan_cost, refuels_per_day, earnings),
            })

results = pd.DataFrame(rows)

# Make 'scenario' an ordered category so sorting and pivots keep favourable -> central -> unfavourable
results["scenario"] = pd.Categorical(results["scenario"], categories=SCENARIOS, ordered=True)
print(f"{len(results)} rows")

54 rows


## 7. Summary: net benefit per day at three earnings levels

**Naira per working day** a driver is better off on CNG (negative = worse off), for each vehicle and scenario at
₦2,000, ₦3,000 and ₦4,000 per hour. Each scenario uses its own queue time (1.5, 4 and 6 hours), so this table
answers "does CNG pay at the queues this scenario assumes?" 

In [7]:
SUMMARY_EARNINGS = [2_000, 3_000, 4_000]

summary = (
    results[results["earnings_per_hour"].isin(SUMMARY_EARNINGS)]
    .pivot_table(index=["vehicle", "scenario"], columns="earnings_per_hour",
                 values="net_benefit", observed=True)
    .reindex(pd.MultiIndex.from_product([list(vehicles), SCENARIOS]))   # vehicle order as defined, not alphabetical
)
summary.index.names = ["Vehicle", "Scenario"]
summary.columns = [f"Net benefit at NGN {e:,}/hr" for e in summary.columns]
summary.style.format("{:,.0f}")

## 8. Central-scenario detail

All money columns are **naira per working day**; break-even is **hours per refuel**. Queue cost uses the central queue
time of 4 hours. Break-even does *not* depend on the actual queue time: it is the queue time that would make net
benefit zero.

In [8]:
central = results[results["scenario"] == "Central"].drop(columns="scenario")

central.rename(columns={
    "vehicle": "Vehicle",
    "earnings_per_hour": "Earnings (NGN/hr)",
    "petrol_spend_per_day": "Petrol spend (NGN/day)",
    "fuel_saving": "Fuel saving (NGN/day)",
    "queue_cost": "Queue cost (NGN/day)",
    "loan_cost": "Loan cost (NGN/day)",
    "net_benefit": "Net benefit (NGN/day)",
    "break_even_queue_hours": "Break-even queue (hrs)",
}).set_index(["Vehicle", "Earnings (NGN/hr)"])

Petrol spend (NGN/day)  \
Vehicle          Earnings (NGN/hr)                           
Ride-hailing car 1000                            28,000.00   
                 1500                            28,000.00   
                 2000                            28,000.00   
                 2500                            28,000.00   
                 3000                            28,000.00   
                 3500                            28,000.00   
                 4000                            28,000.00   
                 4500                            28,000.00   
                 5000                            28,000.00   
Keke             1000                            21,000.00   
                 1500                            21,000.00   
                 2000                            21,000.00   
                 2500                            21,000.00   
                 3000                            21,000.00   
                 3500                            21,000.00   
                 4000                            21,000.00   
                 4500                            21,000.00   
                 5000                            21,000.00   

                                    Fuel saving (NGN/day)  \
Vehicle          Earnings (NGN/hr)                          
Ride-hailing car 1000                           21,640.00   
                 1500                           21,640.00   
                 2000                           21,640.00   
                 2500                           21,640.00   
                 3000                           21,640.00   
                 3500                           21,640.00   
                 4000                           21,640.00   
                 4500                           21,640.00   
                 5000                           21,640.00   
Keke             1000                           16,230.00   
                 1500                           16,230.00   
                 2000                           16,230.00   
                 2500                           16,230.00   
                 3000                           16,230.00   
                 3500                           16,230.00   
                 4000                           16,230.00   
                 4500                           16,230.00   
                 5000                           16,230.00   

                                    Queue cost (NGN/day)  Loan cost (NGN/day)  \
Vehicle          Earnings (NGN/hr)                                              
Ride-hailing car 1000                           4,000.00             2,484.14   
                 1500                           6,000.00             2,484.14   
                 2000                           8,000.00             2,484.14   
                 2500                          10,000.00             2,484.14   
                 3000                          12,000.00             2,484.14   
                 3500                          14,000.00             2,484.14   
                 4000                          16,000.00             2,484.14   
                 4500                          18,000.00             2,484.14   
                 5000                          20,000.00             2,484.14   
Keke             1000                           4,000.00               716.58   
                 1500                           6,000.00               716.58   
                 2000                           8,000.00               716.58   
                 2500                          10,000.00               716.58   
                 3000                          12,000.00               716.58   
                 3500                          14,000.00               716.58   
                 4000                          16,000.00               716.58   
                 4500                          18,000.00               716.58   
                 5000              

### Break-even queue time across all three scenarios

The chart below shows only the central scenario, so this table shows how far the answer moves between the bookends.
Hours per refuel.

In [9]:
results.pivot_table(
    index="earnings_per_hour",
    columns=["vehicle", "scenario"],
    values="break_even_queue_hours",
    observed=True,
)[[(v, s) for v in vehicles for s in SCENARIOS]]   # vehicle and scenario order as defined

vehicle           Ride-hailing car                                    Keke  \
scenario            CNG-favourable Central CNG-unfavourable CNG-favourable   
earnings_per_hour                                                            
1000                         26.92   19.16             7.36          20.36   
1500                         17.95   12.77             4.91          13.57   
2000                         13.46    9.58             3.68          10.18   
2500                         10.77    7.66             2.95           8.14   
3000                          8.97    6.39             2.45           6.79   
3500                          7.69    5.47             2.10           5.82   
4000                          6.73    4.79             1.84           5.09   
4500                          5.98    4.26             1.64           4.52   
5000                          5.38    3.83             1.47           4.07   

vehicle                                     
scenario          Central CNG-unfavourable  
earnings_per_hour                           
1000                15.51             7.75  
1500                10.34             5.17  
2000                 7.76             3.87  
2500                 6.21             3.10  
3000                 5.17             2.58  
3500                 4.43             2.21  
4000                 3.88             1.94  
4500                 3.45             1.72  
5000                 3.10             1.55

## 9. Chart: break-even queue time vs earnings

- **One line per vehicle, central scenario.** Blue = ride-hailing car, orange = keke, the same palette as notebook 01.
- **Grey band = reported queue times**, 1.5 hours (NESG estimate, S02) to 6 hours (Abuja reports, S04), read from the
  `queue_time_per_refuel` row. Where a line sits *above* the band, CNG pays off even at the worst reported queues;
  where it drops *into* the band, real queues can wipe out the gain.
- `add_hrect` draws the band as a shape spanning the full width, with `layer="below"` so the lines sit on top.
- **y-axis starts at 0** so the distance between a line and the band isn't distorted.

In [10]:
COLORS = {"Ride-hailing car": "#2a78d6", "Keke": "#eb6834"}
LABEL_SHIFT = {"Ride-hailing car": 8, "Keke": -8}   # pixels
INK = "#0b0b0b"
INK_MUTED = "#52514e"
BAND_LOW, BAND_HIGH = queue_time["low"], queue_time["high"]   # 1.5 and 6 hours (S02; S04)
c = scenarios["Central"]   # short name for the subtitle and footnote

fig = go.Figure()

fig.add_hrect(
    y0=BAND_LOW, y1=BAND_HIGH,
    fillcolor="#8a8984", opacity=0.15, line_width=0, layer="below",
    annotation_text=f"Reported queue times (S02, S04): {BAND_LOW:g}–{BAND_HIGH:g} hrs",
    annotation_position="top left",
    annotation_font=dict(color=INK_MUTED, size=12),
)

for vehicle_name in vehicles:
    d = central[central["vehicle"] == vehicle_name]
    fig.add_trace(go.Scatter(
        x=d["earnings_per_hour"],
        y=d["break_even_queue_hours"],
        name=vehicle_name,
        mode="lines+markers",
        line=dict(color=COLORS[vehicle_name], width=2),
        marker=dict(size=8, color=COLORS[vehicle_name], line=dict(color="white", width=2)),
        customdata=d[["fuel_saving", "loan_cost"]],
        hovertemplate=(
            f"<b>{vehicle_name}</b><br>"
            "Earnings: ₦%{x:,.0f} per hour<br>"
            "Break-even queue: <b>%{y:.1f} hours</b> per refuel<br>"
            "Fuel saving: ₦%{customdata[0]:,.0f}/day · Loan: ₦%{customdata[1]:,.0f}/day"
            "<extra></extra>"
        ),
    ))
    # Direct label at the right end of each line. The two lines end close together,
    # so nudge the upper one up and the lower one down to keep the labels apart.
    last = d.iloc[-1]
    fig.add_annotation(
        x=last["earnings_per_hour"], y=last["break_even_queue_hours"],
        text=f"{vehicle_name}: {last['break_even_queue_hours']:.1f} hrs",
        xanchor="left", xshift=10, yshift=LABEL_SHIFT[vehicle_name], showarrow=False,
        font=dict(color=INK, size=12),
    )

fig.update_layout(
    title=dict(
        text="How long can a driver queue before CNG stops paying off?<br>"
             f"<sup>Central scenario: petrol ₦{c['petrol_price']:,.0f}/L, CNG ₦{c['cng_price']:,.0f}/scm, "
             f"{c['interest_rate']:.1%} loan over {c['tenor_years']:g} years, {refuels_per_day:g} refuel/day. "
             "Earnings per hour is a sensitivity range pending survey data.</sup>",
        x=0, xanchor="left",
    ),
    template="plotly_white",
    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color=INK),
    xaxis=dict(
        title=dict(text="Driver net earnings (₦ per hour) — assumed range, pending survey", standoff=8),
        tickprefix="₦", tickformat=",.0f", dtick=500, showgrid=False,
        range=[EARNINGS_PER_HOUR[0] - 200, EARNINGS_PER_HOUR[-1] + 200],   # a little padding, no empty tick past 5,000
    ),
    yaxis=dict(
        title=dict(text="Break-even queue time (hours per refuel)", standoff=12),
        rangemode="tozero", gridcolor="#ececea", ticksuffix=" h",
    ),
    legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0, xanchor="left"),
    hovermode="closest",
    margin=dict(t=130, l=80, r=170, b=140),
    height=620,
)

fig.add_annotation(
    text="Break-even = (fuel saving − loan cost) ÷ (refuels per day × earnings per hour). "
         "Central (mid) inputs from data/raw/model_inputs.csv.<br>"
         "Sources (docs/sources.md): petrol & CNG price S11; loan S18; queue band S02, S04; "
         "petrol spend S21 (car), S19 (keke); conversion S40 (car), S43/S04 (keke).<br>"
         f"Petrol spend rescaled from ~₦{spend_ref_price:,.0f}/L to ₦{c['petrol_price']:,.0f}/L "
         f"assuming the same litres per day. Keke conversion ₦{conv_keke['mid']:,.0f} = midpoint of "
         f"₦{conv_keke['low']:,.0f}–₦{conv_keke['high']:,.0f}. Working days {working_days:g}/yr (assumption).",
    xref="paper", yref="paper", x=0, y=-0.25, showarrow=False,
    xanchor="left", align="left", font=dict(size=11, color=INK_MUTED),
)

fig.show()

In [11]:
out_path = FIG_DIR / "break_even_queue.html"
fig.write_html(out_path, include_plotlyjs="cdn")   # cdn: small file, needs internet to render (see notebook 01)
print(f"Saved {out_path.relative_to(PROJECT_ROOT)}")

Saved outputs\figures\break_even_queue.html


## 10. Numbers for the write-up

For every vehicle and scenario: break-even at the two ends of the earnings range, and the earnings level from which
break-even drops to or below 6 hours (the worst reported queue). These feed the explanation below.

In [12]:
for vehicle_name in vehicles:
    print(vehicle_name)
    for scenario_name in SCENARIOS:
        d = (results[(results["vehicle"] == vehicle_name) & (results["scenario"] == scenario_name)]
             .set_index("earnings_per_hour")["break_even_queue_hours"])
        inside = d[d <= BAND_HIGH]
        band_text = (f"enters band from ₦{inside.index.min():,}/hr" if len(inside)
                     else "stays above band")
        print(f"  {scenario_name:<17s} ₦1,000/hr: {d.iloc[0]:5.1f} hrs | ₦5,000/hr: {d.iloc[-1]:4.1f} hrs | {band_text}")

Ride-hailing car
  CNG-favourable    ₦1,000/hr:  26.9 hrs | ₦5,000/hr:  5.4 hrs | enters band from ₦4,500/hr
  Central           ₦1,000/hr:  19.2 hrs | ₦5,000/hr:  3.8 hrs | enters band from ₦3,500/hr
  CNG-unfavourable  ₦1,000/hr:   7.4 hrs | ₦5,000/hr:  1.5 hrs | enters band from ₦1,500/hr
Keke
  CNG-favourable    ₦1,000/hr:  20.4 hrs | ₦5,000/hr:  4.1 hrs | enters band from ₦3,500/hr
  Central           ₦1,000/hr:  15.5 hrs | ₦5,000/hr:  3.1 hrs | enters band from ₦3,000/hr
  CNG-unfavourable  ₦1,000/hr:   7.7 hrs | ₦5,000/hr:  1.5 hrs | enters band from ₦1,500/hr


## 11. What the results mean

**Three scenarios, three different answers.** Which one holds depends mostly on queue time and loan terms, not on the
fuel saving, which is large in all three.

**CNG-favourable (petrol ₦1,596/L, CNG ₦230/scm, cheapest kit, 15% over 3 years, 1.5-hour queue).** CNG pays off
comfortably for both vehicles at every earnings level we tested: about ₦21,000–24,000 a day for the car and
₦14,000–17,000 for the keke at ₦2,000–4,000/hour. Break-even never falls below 4 hours, far above the 1.5-hour queue
this scenario assumes.

**Central (mid values, 4-hour queue).** CNG pays at lower earnings but the margin shrinks fast. The car's break-even
drops into the reported 1.5–6 hour band from ₦3,500/hour and the keke's from ₦3,000/hour. At ₦4,000/hour the car
clears about ₦3,000 a day, while the keke already loses about ₦500 a day.

**CNG-unfavourable (petrol ₦1,051/L, CNG ₦380/scm, dearest kit, 20% over 1 year, 6-hour queue).** CNG loses money for
both vehicles at ₦2,000, ₦3,000 and ₦4,000/hour: between about ₦4,000 and ₦17,000 a day. Break-even falls below 6 hours
from just ₦1,500/hour, so with 6-hour queues only the lowest earners in our range come out ahead. A one-year loan on
an expensive kit makes daily repayments heavy. Here the keke edges slightly ahead of the car, because its kit costs
far less to repay.

**What it means for drivers and policy:** fuel prices alone make CNG attractive (notebook 01). In this model, whether
a driver actually gains depends on **how long they queue and how they pay for the kit**. Policy can influence both,
through station capacity and conversion finance, while petrol and gas prices are harder to steer. The favourable and
unfavourable scenarios are bookends: every input sits at its extreme at once, which is unlikely, so real drivers will
mostly fall between them. The survey will show where.

**Main assumptions, all to be replaced by survey data:**
- **Earnings per hour** (₦1,000–5,000) is a sensitivity range, not an estimate. It decides where on these lines real
  drivers sit.
- **1 refuel per day** and **efficiency ratio 1.0** (1 scm goes as far as 1 litre; calibrated on car data and applied
  to keke, C4). Both are rows in `model_inputs.csv`.
- **Petrol spend rescaled assuming the same litres per day** as when it was reported at about ₦1,000/litre
  (`petrol_price_at_spend_reports`, an Assumption row citing S19, S21).
- **Keke central conversion cost ₦375,000** is the midpoint of the ₦100k–₦650k range, and **312 working days a year**
  is an assumption.
- **The conversion is loan-financed.** A driver who pays cash, or gets a free kit, has no loan cost and can tolerate
  longer queues.
